# Recipe R-MOD-001: Simulasi Difusi Panas Satu Dimensi
Berdasarkan *AI Cookbook untuk Earth Sciences* yang dilampirkan, recipe ini mengimplementasikan pemodelan difusi panas satu dimensi menggunakan metode *explicit finite difference*.

## Teori Minimal (E-HEAT-001)
Persamaan difusi panas:
$$ \frac{\partial T}{\partial t} = \kappa \frac{\partial^2 T}{\partial z^2} $$
dengan $T$ (suhu), $z$ (kedalaman), $t$ (waktu), dan $\kappa$ (difusivitas termal).
Syarat kestabilan metode eksplisit adalah $r = \kappa \Delta t / \Delta z^2 \le 1/2$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameter Model (Berdasarkan Recipe)
kappa = 1.0e-6  # Thermal diffusivity (m^2/s)
z_max = 5.0     # Kedalaman maksimum (m)
t_max = 30 * 24 * 3600  # Durasi simulasi (30 hari dalam detik)

dz = 0.1  # Resolusi spasial (m)
dt = 3600 # Resolusi waktu (detik) - 1 jam

# Cek kestabilan
r = kappa * dt / (dz**2)
print(f"Rasio kestabilan (r): {r:.4f}")
if r > 0.5:
    print("PERINGATAN: Kriteria kestabilan tidak terpenuhi (r > 0.5)!")

## Solusi Numerik
Proses iterasi waktu dan ruang menggunakan metode *finite difference*.

In [ ]:
# Inisialisasi Grid
z = np.arange(0, z_max + dz, dz)
t = np.arange(0, t_max + dt, dt)

nz = len(z)
nt = len(t)

# Matriks Suhu T[time, depth]
T = np.zeros((nt, nz))

# Kondisi Awal (Initial Condition): Suhu awal 10 Celcius di semua kedalaman
T[0, :] = 10.0

# Kondisi Batas (Boundary Conditions)
# Suhu permukaan (z=0) naik menjadi 25 Celcius (Gangguan panas dari permukaan)
T[:, 0] = 25.0
# Suhu dasar (z=z_max) tetap 10 Celcius
T[:, -1] = 10.0

# Simulasi Finite Difference (Eksplisit)
for n in range(0, nt - 1):
    for i in range(1, nz - 1):
        T[n+1, i] = T[n, i] + r * (T[n, i+1] - 2*T[n, i] + T[n, i-1])
        
print("Simulasi finite difference selesai.")

## Visualisasi
Visualisasi hasil akhir simulasi evolusi suhu terhadap kedalaman dan waktu dalam bentuk *heatmap*, seperti yang diinstruksikan dalam Prompt P-MOD-001.

In [ ]:
# Visualisasi Heatmap
plt.figure(figsize=(10, 6))
plt.imshow(T.T, aspect='auto', cmap='inferno', extent=[0, t_max/(24*3600), z_max, 0])
plt.colorbar(label='Temperature (°C)')
plt.xlabel('Time (days)')
plt.ylabel('Depth (m)')
plt.title('One-dimensional subsurface heat diffusion')
plt.show()